In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="05-harmful-content-detection",
    notebook_name="02_multimodal_fusion_and_features.ipynb"
)

# Multimodal Fusion & Feature Engineering

## What We Cover in This Notebook

In the previous notebook, we designed the overall system. Now we are going **deep** into the two core engineering challenges:

1. **Feature Engineering** -- How do we transform raw posts (text, images, videos, metadata) into numbers that ML models can eat?
2. **Multimodal Fusion** -- How do we combine features from different modalities into one unified representation?

Think of it like cooking a complex dish. Notebook 1 was the recipe overview. This notebook is about the actual prep work -- washing, chopping, measuring, and mixing the ingredients. Get this wrong, and the dish (model) will taste terrible no matter how good your oven (training) is.

### The Five Feature Categories

```
                      All Features Combined
                     /    |      |      |    \
                    /     |      |      |     \
              Textual  Image/  User    Author  Context
              Content  Video  Reactions  Info   Info
```

Each category requires different processing. Let's build each one from scratch.

---

## 1. Textual Content Features

### The Big Idea

Text is just letters and words. But ML models only understand **numbers**. So we need to convert text into a vector of numbers that captures the **meaning** of the text.

Think of it like translating a book into sheet music. The original book is in English, but we need it in a language that our ML model (the musician) can play. The translation should preserve the **feeling** of the original -- angry text should produce an "angry-sounding" number sequence.

### Why DistilBERT Over BERT?

| Feature | BERT | DistilBERT (Our Choice) | DistilmBERT |
|---------|------|------------------------|-------------|
| Parameters | 110M | 66M (40% smaller) | 135M |
| Inference speed | Slow | 60% faster | Fast |
| Languages | English only | English only | 100+ languages |
| Embedding quality | Excellent | 97% of BERT | Good across languages |
| Production ready? | Heavy | YES | YES (multilingual) |

**Our choice: DistilmBERT** -- because posts come in many languages, and we need fast inference at 500M posts/day.

**Key interview phrase**: *"We use DistilmBERT over BERT because (1) it is 60% faster for inference -- critical at 500M posts/day, and (2) it supports 100+ languages out of the box since it was trained on multilingual data."*

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# ======================================================================
# Text Feature Extraction with a Simulated DistilBERT
# 
# In production, you would use:
#   from transformers import DistilBertModel, DistilBertTokenizer
#   model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
#
# Here we simulate the full pipeline to show you how it works
# without needing to download the actual model (which is ~500MB).
# ======================================================================

class SimpleTokenizer:
    """
    A simplified tokenizer that shows how text gets converted to token IDs.
    
    Think of it like a dictionary where each word has an ID number.
    The sentence 'The cat sat' becomes [2, 45, 102] (word IDs).
    
    Real tokenizers (like WordPiece used in BERT) also split unknown 
    words into sub-words: 'unhappiness' -> ['un', '##happi', '##ness']
    """
    
    def __init__(self, vocab_size=30522, max_length=128):
        self.vocab_size = vocab_size
        self.max_length = max_length
        # Special tokens
        self.cls_token_id = 101  # [CLS] - used as the sentence embedding
        self.sep_token_id = 102  # [SEP] - marks end of sentence
        self.pad_token_id = 0    # [PAD] - padding for shorter sequences
    
    def tokenize(self, text):
        """Convert text to token IDs (simplified)."""
        # Real tokenizer: WordPiece subword tokenization
        # Our simplified version: hash each word to a vocab ID
        words = text.lower().split()
        token_ids = [self.cls_token_id]  # Start with [CLS]
        for word in words[:self.max_length - 2]:  # Leave room for [CLS] and [SEP]
            token_ids.append(hash(word) % (self.vocab_size - 3) + 3)  # Avoid special tokens
        token_ids.append(self.sep_token_id)  # End with [SEP]
        
        # Pad to max_length
        attention_mask = [1] * len(token_ids) + [0] * (self.max_length - len(token_ids))
        token_ids = token_ids + [self.pad_token_id] * (self.max_length - len(token_ids))
        
        return {
            'input_ids': torch.tensor(token_ids),
            'attention_mask': torch.tensor(attention_mask)
        }


class SimulatedDistilBERT(nn.Module):
    """
    Simulates DistilBERT's architecture for educational purposes.
    
    The real DistilBERT:
    1. Embeds each token into a 768-dim vector
    2. Passes through 6 Transformer layers (attention + feedforward)
    3. Outputs a 768-dim vector for EACH token position
    4. We take the [CLS] token's output as the SENTENCE embedding
    
    Think of it like a reading comprehension test:
    - Each word gets its own understanding (token embeddings)
    - The Transformer layers are like re-reading the sentence multiple times,
      each time understanding more context
    - The [CLS] token at the end "summarizes" the whole sentence
    """
    
    def __init__(self, vocab_size=30522, hidden_size=768, max_length=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.position_embedding = nn.Embedding(max_length, hidden_size)
        
        # Simplified Transformer layer (real DistilBERT has 6)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size, nhead=12, dim_feedforward=3072,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.hidden_size = hidden_size
    
    def forward(self, input_ids, attention_mask=None):
        seq_length = input_ids.size(1)
        position_ids = torch.arange(seq_length).unsqueeze(0).expand_as(input_ids)
        
        # Step 1: Embed tokens + positions
        x = self.embedding(input_ids) + self.position_embedding(position_ids)
        
        # Step 2: Pass through Transformer layers
        if attention_mask is not None:
            # Convert to the format Transformer expects
            src_key_padding_mask = (attention_mask == 0)
            x = self.transformer(x, src_key_padding_mask=src_key_padding_mask)
        else:
            x = self.transformer(x)
        
        # Step 3: Take the [CLS] token output as the sentence embedding
        cls_output = x[:, 0, :]  # First token is [CLS]
        
        return cls_output  # Shape: (batch_size, 768)


# ---- Demo ----
torch.manual_seed(42)
tokenizer = SimpleTokenizer()
text_model = SimulatedDistilBERT()

# Example posts
posts = [
    "I hate everyone in this group",
    "Beautiful sunset over the mountains",
    "Watch this violent fight video",
]

print("=== Text Feature Extraction Pipeline ===")
print()

text_embeddings = []
for text in posts:
    # Step 1: Tokenize
    tokens = tokenizer.tokenize(text)
    print(f"Text: '{text}'")
    print(f"  Token IDs (first 10): {tokens['input_ids'][:10].tolist()}")
    print(f"  Attention mask (first 10): {tokens['attention_mask'][:10].tolist()}")
    
    # Step 2: Get embedding
    with torch.no_grad():
        embedding = text_model(tokens['input_ids'].unsqueeze(0), tokens['attention_mask'].unsqueeze(0))
    
    text_embeddings.append(embedding.squeeze(0))
    print(f"  Embedding shape: {embedding.shape} (768-dimensional vector)")
    print(f"  Embedding sample: [{embedding[0, :5].tolist()}...]")
    print()

# Show that different texts produce different embeddings
from torch.nn.functional import cosine_similarity
print("Cosine similarities between text embeddings:")
for i in range(len(posts)):
    for j in range(i+1, len(posts)):
        sim = cosine_similarity(text_embeddings[i].unsqueeze(0), text_embeddings[j].unsqueeze(0))
        print(f"  '{posts[i][:30]}...' vs '{posts[j][:30]}...': {sim.item():.4f}")

---

## 2. Image and Video Features

### The Big Idea

Just like text, images are raw data (millions of pixel values) that need to be converted into a compact, meaningful vector. We use pre-trained visual models to do this.

Think of it like hiring an art expert to describe a painting. The raw painting has millions of individual brushstrokes (pixels), but the expert condenses it into a few key descriptors: "landscape, sunset, warm colors, peaceful mood" -- that is the embedding.

### Model Options

| Model | What It Learns | Best For | Embedding Size |
|-------|---------------|----------|----------------|
| **CLIP (Visual Encoder)** | Image-text alignment | Meme detection (cross-modal!) | 512 |
| **SimCLR** | Self-supervised visual features | General image understanding | 2048 -> projected |
| **VideoMoCo** | Video-level features from frames | Video content analysis | Varies |

**Key interview insight**: CLIP is particularly powerful for harmful content detection because it was trained to understand the relationship between images and text -- exactly what we need for detecting harmful memes.

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
import numpy as np

# ======================================================================
# Image Feature Extraction Pipeline
# 
# Two steps, just like a chef preparing ingredients:
# 1. Preprocessing: Wash, peel, chop (resize, normalize)
# 2. Feature extraction: Cook into final form (pass through CNN)
# ======================================================================

# Step 1: Preprocessing pipeline
image_preprocess = transforms.Compose([
    transforms.Resize(256),            # Resize shorter side to 256
    transforms.CenterCrop(224),        # Crop center 224x224
    transforms.ToTensor(),             # [0,255] -> [0,1], HWC -> CHW
    transforms.Normalize(              # Z-score normalization
        mean=[0.485, 0.456, 0.406],    # ImageNet means
        std=[0.229, 0.224, 0.225]      # ImageNet stds
    ),
])

# Step 2: Simulated image encoder (simplified CLIP visual encoder)
class SimulatedCLIPVisualEncoder(nn.Module):
    """
    Simulates CLIP's visual encoder.
    
    The real CLIP uses a Vision Transformer (ViT) or ResNet to:
    1. Break the image into patches (like puzzle pieces)
    2. Process each patch through Transformer layers
    3. Output a 512-dim embedding that captures the image's content
    
    The magic of CLIP: it was trained on 400M image-text pairs,
    so it understands the RELATIONSHIP between images and text.
    This is perfect for detecting harmful memes!
    """
    
    def __init__(self, embedding_dim=512):
        super().__init__()
        # Simplified backbone (real CLIP uses ViT-B/32 or ResNet-50x4)
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((7, 7)),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.projection = nn.Linear(128, embedding_dim)
    
    def forward(self, x):
        features = self.conv_layers(x)
        features = features.flatten(1)  # (batch, 128)
        embedding = self.projection(features)  # (batch, 512)
        embedding = nn.functional.normalize(embedding, p=2, dim=1)  # L2 normalize
        return embedding


# ---- Demo ----
torch.manual_seed(42)
image_model = SimulatedCLIPVisualEncoder(embedding_dim=512)

# Create a synthetic image (in production, you load real images)
dummy_image = Image.fromarray(
    np.random.randint(0, 255, (300, 400, 3), dtype=np.uint8)
).convert('RGB')

# Process it
preprocessed = image_preprocess(dummy_image).unsqueeze(0)  # Add batch dimension

with torch.no_grad():
    image_embedding = image_model(preprocessed)

print("=== Image Feature Extraction ===")
print(f"Raw image shape:        {np.array(dummy_image).shape}  (H=300, W=400, C=3)")
print(f"After preprocessing:    {preprocessed.shape}      (B=1, C=3, H=224, W=224)")
print(f"Image embedding shape:  {image_embedding.shape}      (B=1, D=512)")
print(f"Embedding norm:         {image_embedding.norm().item():.4f}  (unit-normalized)")
print(f"\nEach image is now a 512-dimensional vector.")
print(f"Total compression: {300*400*3:,} pixels -> 512 numbers = {300*400*3/512:.0f}x compression!")

In [ ]:
# ======================================================================
# Video Feature Extraction
# 
# Videos are just sequences of images (frames). We have two choices:
# 1. Sample key frames and average their embeddings (simple)
# 2. Use a video-specific model like VideoMoCo (better but heavier)
#
# Think of it like summarizing a movie:
# Option 1: Look at a few key screenshots and describe the movie
# Option 2: Watch the whole movie and write a review
# ======================================================================

def extract_video_features(video_frames, image_model, num_samples=8):
    """
    Extract features from a video by sampling and averaging frame embeddings.
    
    Args:
        video_frames: List of PIL images (frames)
        image_model: Pre-trained image encoder
        num_samples: Number of frames to sample (uniformly spaced)
    
    Returns:
        video_embedding: (512,) averaged frame embedding
    """
    total_frames = len(video_frames)
    
    # Uniformly sample frames
    indices = np.linspace(0, total_frames - 1, num_samples, dtype=int)
    sampled_frames = [video_frames[i] for i in indices]
    
    # Preprocess and embed each frame
    frame_embeddings = []
    for frame in sampled_frames:
        preprocessed = image_preprocess(frame).unsqueeze(0)
        with torch.no_grad():
            embedding = image_model(preprocessed)
        frame_embeddings.append(embedding)
    
    # Average all frame embeddings
    stacked = torch.cat(frame_embeddings, dim=0)  # (num_samples, 512)
    video_embedding = stacked.mean(dim=0, keepdim=True)  # (1, 512)
    
    # Re-normalize
    video_embedding = nn.functional.normalize(video_embedding, p=2, dim=1)
    
    return video_embedding


# ---- Demo ----
# Simulate a 30-frame video
fake_video = [Image.fromarray(
    np.random.randint(0, 255, (240, 320, 3), dtype=np.uint8)
).convert('RGB') for _ in range(30)]

video_emb = extract_video_features(fake_video, image_model, num_samples=8)

print("=== Video Feature Extraction ===")
print(f"Video: {len(fake_video)} frames")
print(f"Sampled: 8 uniformly spaced frames")
print(f"Each frame -> 512-dim embedding")
print(f"Average across frames -> 1 video embedding")
print(f"Final video embedding shape: {video_emb.shape}")
print()
print("Why sample frames instead of using all?")
print("  - A 1-minute video at 30fps = 1800 frames")
print("  - Processing all 1800 frames is too expensive")
print("  - 8 uniformly sampled frames capture the key content")
print("  - For violent content, key frames are often enough")

---

## 3. User Reaction Features

### The Big Idea

Sometimes the post itself looks innocent, but the **community reaction** reveals it is harmful. Think of it like a smoke detector. The post is a room -- you might not see a fire, but if smoke starts coming out (people reporting, angry comments), something is wrong.

Over time, as more users interact with a post, the signal gets **stronger**:

```
Time 0: Post created    --> No signal (just content features)
Time 1: 2 comments      --> "This is disturbing" --> Hmm, maybe harmful
Time 2: 5 reports       --> Multiple users flagging --> Probably harmful
Time 3: 50 reports      --> Strong community signal --> Almost certainly harmful
```

This is exactly what Figure 5.13 in the PDF illustrates -- confidence grows over time as reactions accumulate.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# ======================================================================
# Comment Aggregation Pipeline
#
# How do we turn a bunch of text comments into a single feature vector?
# 
# Think of it like reading Amazon reviews:
# 1. Read each review individually (embed each comment)
# 2. Form an overall impression (average the embeddings)
# 
# The "overall impression" vector captures the collective sentiment.
# ======================================================================

def aggregate_comment_embeddings(comments, text_model, tokenizer):
    """
    Convert a list of comments into a single aggregated embedding.
    
    Steps:
    1. Tokenize each comment
    2. Get embedding for each comment (using same text model as post text)
    3. Average all embeddings into one vector
    
    Args:
        comments: List of comment strings
        text_model: Pre-trained text encoder (DistilBERT)
        tokenizer: Tokenizer instance
    
    Returns:
        aggregated_embedding: (768,) vector summarizing all comments
    """
    if not comments:
        return torch.zeros(768)  # Return zero vector if no comments
    
    embeddings = []
    for comment in comments:
        tokens = tokenizer.tokenize(comment)
        with torch.no_grad():
            emb = text_model(
                tokens['input_ids'].unsqueeze(0),
                tokens['attention_mask'].unsqueeze(0)
            )
        embeddings.append(emb.squeeze(0))
    
    # Stack and average
    stacked = torch.stack(embeddings)  # (num_comments, 768)
    aggregated = stacked.mean(dim=0)   # (768,)
    
    return aggregated


# ---- Demo ----
torch.manual_seed(42)

# Reuse models from earlier
tokenizer = SimpleTokenizer()
text_model = SimulatedDistilBERT()
text_model.eval()

# Comments on a suspicious post
comments = [
    "This is really disturbing content",
    "Please remove this immediately",
    "I am reporting this post",
    "This violates community guidelines",
    "Extremely inappropriate",
]

comment_embedding = aggregate_comment_embeddings(comments, text_model, tokenizer)

print("=== Comment Aggregation Pipeline ===")
print(f"Number of comments: {len(comments)}")
print(f"Each comment -> 768-dim embedding")
print(f"Averaged -> Single 768-dim vector")
print(f"Final comment embedding shape: {comment_embedding.shape}")
print()
print("Comments processed:")
for i, c in enumerate(comments, 1):
    print(f"  {i}. '{c}'")
print()
print("The aggregated embedding captures the overall 'angry/concerned' sentiment")
print("of all comments combined -- a strong signal for harmful content!")

In [ ]:
# ======================================================================
# Reaction Count Features
#
# Besides comment embeddings, we also use raw counts:
# - Number of likes, shares, comments, reports
# These are scaled (normalized) to speed up training convergence.
# ======================================================================

import torch
import numpy as np

def compute_reaction_features(interaction_data):
    """
    Compute numerical reaction features from interaction data.
    
    Think of it like a report card with different scores:
    - Lots of likes and few reports = probably safe
    - Lots of reports and angry comments = probably harmful
    - High report-to-impression ratio = suspicious
    """
    features = {
        'num_likes': interaction_data.get('likes', 0),
        'num_shares': interaction_data.get('shares', 0),
        'num_comments': interaction_data.get('comments', 0),
        'num_reports': interaction_data.get('reports', 0),
        'num_impressions': interaction_data.get('impressions', 1),  # Avoid division by zero
    }
    
    # Derived features (ratios are very predictive!)
    features['report_rate'] = features['num_reports'] / max(features['num_impressions'], 1)
    features['engagement_rate'] = (
        features['num_likes'] + features['num_comments'] + features['num_shares']
    ) / max(features['num_impressions'], 1)
    
    # Convert to tensor and log-scale (large counts can dominate)
    raw_values = torch.tensor(list(features.values()), dtype=torch.float32)
    scaled_values = torch.log1p(raw_values)  # log(1 + x) to handle zeros
    
    return scaled_values, features


# ---- Demo ----
# Safe post
safe_interactions = {'likes': 500, 'shares': 100, 'comments': 50, 'reports': 1, 'impressions': 10000}
safe_features, safe_dict = compute_reaction_features(safe_interactions)

# Harmful post
harmful_interactions = {'likes': 10, 'shares': 5, 'comments': 200, 'reports': 150, 'impressions': 5000}
harmful_features, harmful_dict = compute_reaction_features(harmful_interactions)

print("=== Reaction Features Comparison ===")
print(f"{'Feature':<20} {'Safe Post':>12} {'Harmful Post':>12}")
print("-" * 44)
for key in safe_dict:
    print(f"{key:<20} {safe_dict[key]:>12.4f} {harmful_dict[key]:>12.4f}")

print(f"\nScaled feature vectors:")
print(f"  Safe:    {safe_features.tolist()[:5]}")
print(f"  Harmful: {harmful_features.tolist()[:5]}")
print()
print("Key insight: The report_rate is 100x higher for the harmful post!")
print(f"  Safe: {safe_dict['report_rate']:.6f}")
print(f"  Harmful: {harmful_dict['report_rate']:.6f}")

In [ ]:
# Visualize how confidence grows over time as reactions accumulate
# This corresponds to Figure 5.13 in the PDF

import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)

# Simulate a harmful post's confidence over time
time_hours = np.arange(0, 25)
cumulative_reports = np.cumsum(np.random.poisson(3, size=25))  # Reports arrive
cumulative_comments = np.cumsum(np.random.poisson(5, size=25))  # Comments arrive

# Model confidence increases with more signals (sigmoid-like curve)
content_score = 0.35  # Initial content-only score
confidence = content_score + (1 - content_score) * (1 - np.exp(-cumulative_reports / 20))
confidence = np.clip(confidence, 0, 0.99)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: Cumulative reactions
ax1.plot(time_hours, cumulative_reports, 'r-o', linewidth=2, markersize=5, label='Reports')
ax1.plot(time_hours, cumulative_comments, 'b-s', linewidth=2, markersize=5, label='Comments')
ax1.set_xlabel('Hours Since Post Created', fontsize=12)
ax1.set_ylabel('Cumulative Count', fontsize=12)
ax1.set_title('User Reactions Over Time', fontsize=14, fontweight='bold')
ax1.legend(fontsize=12)
ax1.grid(True, alpha=0.3)

# Right: Confidence over time
ax2.plot(time_hours, confidence, 'g-o', linewidth=2.5, markersize=5, label='Model Confidence')
ax2.axhline(y=0.8, color='red', linestyle='--', alpha=0.7, label='Remove Threshold (0.8)')
ax2.axhline(y=0.4, color='orange', linestyle='--', alpha=0.7, label='Demote Threshold (0.4)')
ax2.axhline(y=content_score, color='gray', linestyle=':', alpha=0.7, label=f'Content-only Score ({content_score})')
ax2.set_xlabel('Hours Since Post Created', fontsize=12)
ax2.set_ylabel('Harm Probability', fontsize=12)
ax2.set_title('Confidence Grows with Reactions', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10, loc='lower right')
ax2.set_ylim(0, 1.0)
ax2.grid(True, alpha=0.3)

# Mark the crossing points
demote_hour = np.argmax(confidence >= 0.4)
remove_hour = np.argmax(confidence >= 0.8)
ax2.annotate(f'Demoted at hour {demote_hour}', xy=(demote_hour, 0.4), 
             xytext=(demote_hour + 3, 0.25),
             fontsize=10, fontweight='bold', color='orange',
             arrowprops=dict(arrowstyle='->', color='orange'))
if remove_hour > 0:
    ax2.annotate(f'Removed at hour {remove_hour}', xy=(remove_hour, 0.8), 
                 xytext=(remove_hour + 3, 0.65),
                 fontsize=10, fontweight='bold', color='red',
                 arrowprops=dict(arrowstyle='->', color='red'))

plt.suptitle('How User Reactions Boost Harmful Content Detection Confidence',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("At post creation (hour 0): Only content-based features are available (score = 0.35)")
print(f"After {demote_hour} hours: Enough reports/comments to DEMOTE the post")
print(f"After {remove_hour} hours: Strong enough signal to REMOVE the post")
print("\nThis is why user reactions are so valuable -- they provide 'natural labels' for free!")

---

## 4. Author Features

The person who posted something tells us a lot about whether it might be harmful. Think of it like a background check:

- **Repeat offenders** (many past violations) are more likely to post harmful content
- **New accounts** (< 30 days old) are often spam or malicious
- **Profanity rate** in past posts/comments is predictive
- **Demographics** (age, location) provide context but must be used carefully to avoid bias

### Feature Types and Encoding

| Feature | Type | Encoding Method |
|---------|------|----------------|
| Number of violations | Numerical | Log-scale |
| Total user reports received | Numerical | Log-scale |
| Profane word rate | Numerical | Direct (already a rate) |
| Age | Numerical | Direct or bucketed |
| Gender | Categorical (few values) | One-hot encoding |
| City, Country | Categorical (many values) | Embedding layer |
| Number of followers/followings | Numerical | Log-scale |
| Account age | Numerical | Log-scale |

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# ======================================================================
# Author Feature Encoder
#
# Handles three types of features differently:
# 1. Numerical features -> log-scale + standardize
# 2. Low-cardinality categorical -> one-hot encoding (gender, device)
# 3. High-cardinality categorical -> learned embedding (city, country)
#
# Think of it like a resume scanner:
# - Numbers (years of experience) -> just normalize them
# - Simple categories (degree level: BS/MS/PhD) -> check the box
# - Complex categories (university name: 1000s of options) -> learn
#   a compact representation
# ======================================================================

class AuthorFeatureEncoder(nn.Module):
    """
    Encodes all author-related features into a single vector.
    
    Why we use embedding layers for city/country instead of one-hot:
    - One-hot for 10,000 cities = 10,000-dim sparse vector (wasteful!)
    - Embedding: learn a 16-dim dense vector per city (compact!)
    - Similar cities (same country, same size) get similar embeddings
    """
    
    def __init__(self, num_cities=10000, num_countries=200, 
                 city_emb_dim=16, country_emb_dim=8):
        super().__init__()
        # Embedding layers for high-cardinality categoricals
        self.city_embedding = nn.Embedding(num_cities, city_emb_dim)
        self.country_embedding = nn.Embedding(num_countries, country_emb_dim)
        
        # Output dimension calculation:
        # 5 numerical features (violations, reports, profanity_rate, age, account_age)
        # 2 numerical features (followers, followings)
        # 3 one-hot features (gender: M/F/Other)
        # city_emb_dim + country_emb_dim
        self.output_dim = 7 + 3 + city_emb_dim + country_emb_dim  # = 34
    
    def forward(self, numerical_features, gender_onehot, city_ids, country_ids):
        """
        Args:
            numerical_features: (batch, 7) - violations, reports, profanity_rate, age, 
                                             account_age, followers, followings
            gender_onehot: (batch, 3) - one-hot encoded gender
            city_ids: (batch,) - integer city IDs
            country_ids: (batch,) - integer country IDs
        """
        # Log-scale numerical features (handle zeros with log1p)
        log_numerical = torch.log1p(numerical_features)
        
        # Embed city and country
        city_emb = self.city_embedding(city_ids)      # (batch, 16)
        country_emb = self.country_embedding(country_ids)  # (batch, 8)
        
        # Concatenate everything
        author_features = torch.cat([
            log_numerical,   # (batch, 7)
            gender_onehot,   # (batch, 3)
            city_emb,        # (batch, 16)
            country_emb,     # (batch, 8)
        ], dim=1)            # (batch, 34)
        
        return author_features


# ---- Demo ----
torch.manual_seed(42)
author_encoder = AuthorFeatureEncoder()

# Simulate 2 authors: safe user and risky user
# numerical: [violations, reports, profanity_rate, age, account_age, followers, followings]
numerical = torch.tensor([
    [0, 0, 0.01, 28, 1825, 1200, 500],  # Safe: 0 violations, 5-year account
    [5, 25, 0.15, 16, 30, 50, 20],      # Risky: 5 violations, 1-month account
], dtype=torch.float32)

gender = torch.tensor([
    [1, 0, 0],  # Female
    [0, 1, 0],  # Male
], dtype=torch.float32)

city_ids = torch.tensor([42, 1337])    # San Francisco, some other city
country_ids = torch.tensor([1, 45])    # USA, another country

author_features = author_encoder(numerical, gender, city_ids, country_ids)

print("=== Author Feature Encoding ===")
print(f"\nSafe user:")
print(f"  Violations: 0, Reports: 0, Profanity: 0.01, Account age: 5 years")
print(f"  Feature vector shape: {author_features[0].shape}")
print(f"  Feature vector (first 10): {author_features[0][:10].tolist()}")

print(f"\nRisky user:")
print(f"  Violations: 5, Reports: 25, Profanity: 0.15, Account age: 30 days")
print(f"  Feature vector shape: {author_features[1].shape}")
print(f"  Feature vector (first 10): {author_features[1][:10].tolist()}")

print(f"\nAuthor feature dimension: {author_encoder.output_dim}")
print(f"  = 7 (numerical) + 3 (gender one-hot) + 16 (city emb) + 8 (country emb)")

---

## 5. Contextual Features

Context features capture the **when** and **how** of a post, not the **what**. These are surprisingly predictive:

- **Time of day**: Harmful content is posted more frequently late at night
- **Device type**: Certain behaviors correlate with device (e.g., mass-posting from desktop might indicate a bot)

Think of it like weather patterns. You cannot predict the exact temperature from the time of year, but you know winter is colder than summer. Similarly, late-night posts are statistically more likely to contain harmful content.

In [ ]:
import torch
import numpy as np

def compute_context_features(timestamp_hour, device_type):
    """
    Compute contextual features from post metadata.
    
    Time of day: Bucketed into 5 categories, one-hot encoded
      [morning (6-12), noon (12-14), afternoon (14-18), evening (18-22), night (22-6)]
    
    Device: One-hot encoded
      [smartphone, tablet, desktop]
    """
    # Time buckets
    time_bucket = [0, 0, 0, 0, 0]  # morning, noon, afternoon, evening, night
    if 6 <= timestamp_hour < 12:
        time_bucket[0] = 1   # morning
    elif 12 <= timestamp_hour < 14:
        time_bucket[1] = 1   # noon
    elif 14 <= timestamp_hour < 18:
        time_bucket[2] = 1   # afternoon
    elif 18 <= timestamp_hour < 22:
        time_bucket[3] = 1   # evening
    else:
        time_bucket[4] = 1   # night (22-6)
    
    # Device encoding
    device_map = {'smartphone': [1, 0, 0], 'tablet': [0, 1, 0], 'desktop': [0, 0, 1]}
    device_onehot = device_map.get(device_type, [0, 0, 0])
    
    return torch.tensor(time_bucket + device_onehot, dtype=torch.float32)


# ---- Demo ----
print("=== Context Feature Encoding ===")
test_cases = [
    (8, 'smartphone'),   # Morning post from phone
    (13, 'desktop'),     # Noon post from desktop
    (23, 'smartphone'),  # Late night post from phone
]

time_labels = ['morning', 'noon', 'afternoon', 'evening', 'night']
device_labels = ['smartphone', 'tablet', 'desktop']

for hour, device in test_cases:
    features = compute_context_features(hour, device)
    time_bucket = time_labels[features[:5].argmax().item()]
    print(f"  Hour={hour:2d}, Device={device:<12s} -> time_bucket={time_bucket:<10s} vector={features.tolist()}")

print(f"\nContext feature dimension: 5 (time) + 3 (device) = 8")

---

## 6. The Complete Feature Vector: Concatenation

Now the magic moment -- we combine ALL five feature categories into one giant vector. Think of it like assembling a complete dossier on each post. Every piece of evidence goes into one file.

```
Complete Feature Vector:
[ text_embedding (768) | image_embedding (512) | comment_embedding (768) | 
  reaction_counts (7)  | author_features (34)  | context_features (8)    ]

Total dimension: 768 + 512 + 768 + 7 + 34 + 8 = 2,097
```

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# ======================================================================
# Complete Feature Fusion Module
#
# This is the EARLY FUSION architecture. All features are concatenated
# into one vector before being fed to the multi-task model.
#
# Think of it like assembling a puzzle: each feature category is a
# section of the puzzle. When you put them all together, you can
# finally see the complete picture.
# ======================================================================

class EarlyFusionFeatureExtractor(nn.Module):
    """
    Combines all 5 feature categories into a single vector.
    
    Feature dimensions:
    - Text embedding:     768 (from DistilBERT)
    - Image embedding:    512 (from CLIP)
    - Comment embedding:  768 (from DistilBERT, averaged)
    - Reaction counts:      7 (likes, shares, comments, reports, impressions + derived)
    - Author features:     34 (numerical + one-hot + embeddings)
    - Context features:     8 (time bucket + device)
    - TOTAL:            2,097
    """
    
    def __init__(self):
        super().__init__()
        self.text_dim = 768
        self.image_dim = 512
        self.comment_dim = 768
        self.reaction_dim = 7
        self.author_dim = 34
        self.context_dim = 8
        
        self.total_dim = (
            self.text_dim + self.image_dim + self.comment_dim +
            self.reaction_dim + self.author_dim + self.context_dim
        )
    
    def forward(self, text_emb, image_emb, comment_emb, reaction_features,
                author_features, context_features):
        """
        Concatenate all features into one vector.
        
        Args:
            text_emb: (batch, 768)
            image_emb: (batch, 512)
            comment_emb: (batch, 768)
            reaction_features: (batch, 7)
            author_features: (batch, 34)
            context_features: (batch, 8)
        
        Returns:
            fused: (batch, 2097)
        """
        fused = torch.cat([
            text_emb,
            image_emb,
            comment_emb,
            reaction_features,
            author_features,
            context_features,
        ], dim=1)
        
        return fused


# ---- Demo ----
torch.manual_seed(42)
batch_size = 4

fusioner = EarlyFusionFeatureExtractor()

# Simulate features for a batch of 4 posts
text_emb = torch.randn(batch_size, 768)
image_emb = torch.randn(batch_size, 512)
comment_emb = torch.randn(batch_size, 768)
reaction_feat = torch.randn(batch_size, 7)
author_feat = torch.randn(batch_size, 34)
context_feat = torch.randn(batch_size, 8)

fused = fusioner(text_emb, image_emb, comment_emb, reaction_feat, author_feat, context_feat)

print("=== Early Fusion: Complete Feature Vector ===")
print(f"\nInput dimensions:")
print(f"  Text embedding:       {text_emb.shape[1]:>5} dims  (DistilBERT)")
print(f"  Image embedding:      {image_emb.shape[1]:>5} dims  (CLIP)")
print(f"  Comment embedding:    {comment_emb.shape[1]:>5} dims  (DistilBERT averaged)")
print(f"  Reaction counts:      {reaction_feat.shape[1]:>5} dims  (log-scaled)")
print(f"  Author features:      {author_feat.shape[1]:>5} dims  (mixed encoding)")
print(f"  Context features:     {context_feat.shape[1]:>5} dims  (one-hot)")
print(f"  {'─' * 40}")
print(f"  TOTAL fused vector:   {fused.shape[1]:>5} dims")
print(f"\nBatch output shape: {fused.shape} (4 posts, each with {fused.shape[1]} features)")

In [ ]:
# Visualize the feature vector composition
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Pie chart of feature dimensions
labels = ['Text\n(768)', 'Image\n(512)', 'Comments\n(768)', 
          'Reactions\n(7)', 'Author\n(34)', 'Context\n(8)']
sizes = [768, 512, 768, 7, 34, 8]
colors = ['#2196F3', '#F44336', '#4CAF50', '#FF9800', '#9C27B0', '#607D8B']
explode = (0.02, 0.02, 0.02, 0.08, 0.05, 0.08)  # Emphasize small features

ax1.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
        shadow=True, startangle=90, textprops={'fontsize': 10, 'fontweight': 'bold'})
ax1.set_title('Feature Vector Composition (by Dimension)', fontsize=14, fontweight='bold')

# Bar chart comparing importance vs. size
# (estimated importance -- in practice, use permutation importance or SHAP)
feature_names = ['Text', 'Image', 'Comments', 'Reactions', 'Author', 'Context']
dimensions = [768, 512, 768, 7, 34, 8]
importance = [0.30, 0.25, 0.20, 0.10, 0.10, 0.05]  # Estimated relative importance

x = np.arange(len(feature_names))
width = 0.35

bars1 = ax2.bar(x - width/2, [d/max(dimensions) for d in dimensions], width, 
                label='Relative Dimension', color='#BBDEFB', edgecolor='#1565C0', linewidth=1.5)
bars2 = ax2.bar(x + width/2, [i/max(importance) for i in importance], width,
                label='Estimated Importance', color='#C8E6C9', edgecolor='#2E7D32', linewidth=1.5)

ax2.set_xlabel('Feature Category', fontsize=12)
ax2.set_ylabel('Normalized Scale', fontsize=12)
ax2.set_title('Feature Dimension vs. Importance', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(feature_names, fontsize=10)
ax2.legend(fontsize=11)
ax2.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insight: Reaction features are only 7 dimensions but contribute ~10% of")
print("predictive power! Small features can be very important.")
print("\nText and image embeddings dominate the feature space (95% of dimensions)")
print("because they are the primary content being analyzed for harm.")

---

## 7. Early Fusion vs. Late Fusion: Full Implementation

Now let's implement both fusion architectures in PyTorch and compare them side by side. This is the core architectural decision in the system.

In [ ]:
import torch
import torch.nn as nn

# ======================================================================
# EARLY FUSION MODEL (Our Choice)
#
# All features are concatenated first, then processed by a shared model.
# Like putting all ingredients in one pot and cooking together.
# The model can learn relationships BETWEEN modalities (cross-modal).
# ======================================================================

class EarlyFusionHarmDetector(nn.Module):
    """
    Early fusion: concatenate all features, then classify.
    
    Analogy: A detective who examines ALL evidence simultaneously.
    They can see that the innocent-looking text + innocent-looking image
    together form a harmful meme. A detective who only sees one piece
    of evidence at a time would miss this connection.
    """
    
    def __init__(self, text_dim=768, image_dim=512, metadata_dim=49,
                 hidden_dim=512, num_harm_categories=5):
        super().__init__()
        total_input = text_dim + image_dim + metadata_dim
        
        # Shared layers -- learn cross-modal patterns
        self.shared = nn.Sequential(
            nn.Linear(total_input, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        
        # Task-specific heads
        self.task_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim // 2, 64),
                nn.ReLU(),
                nn.Linear(64, 1),
                nn.Sigmoid(),
            ) for _ in range(num_harm_categories)
        ])
    
    def forward(self, text_emb, image_emb, metadata):
        # EARLY fusion: concatenate all features
        fused = torch.cat([text_emb, image_emb, metadata], dim=1)
        
        # Pass through shared layers
        shared_repr = self.shared(fused)
        
        # Each task head produces its own prediction
        predictions = [head(shared_repr) for head in self.task_heads]
        predictions = torch.cat(predictions, dim=1)  # (batch, num_categories)
        
        return predictions


# ======================================================================
# LATE FUSION MODEL (For Comparison)
#
# Each modality is processed separately, then predictions are combined.
# Like having separate cooks for each ingredient, then mixing the dishes.
# Each cook cannot taste the other's ingredient.
# ======================================================================

class LateFusionHarmDetector(nn.Module):
    """
    Late fusion: separate models per modality, combined at prediction.
    
    Analogy: Three separate detectives, each only seeing one type of evidence.
    They submit independent reports, and a judge combines them.
    None of them can connect evidence across types.
    """
    
    def __init__(self, text_dim=768, image_dim=512, metadata_dim=49,
                 hidden_dim=256, num_harm_categories=5):
        super().__init__()
        
        # Separate model per modality
        self.text_model = nn.Sequential(
            nn.Linear(text_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_harm_categories),
            nn.Sigmoid(),
        )
        
        self.image_model = nn.Sequential(
            nn.Linear(image_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_harm_categories),
            nn.Sigmoid(),
        )
        
        self.metadata_model = nn.Sequential(
            nn.Linear(metadata_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, num_harm_categories),
            nn.Sigmoid(),
        )
        
        # Learnable fusion weights
        self.fusion_weights = nn.Parameter(torch.ones(3) / 3)
    
    def forward(self, text_emb, image_emb, metadata):
        # LATE fusion: separate predictions, then combine
        text_pred = self.text_model(text_emb)        # (batch, num_categories)
        image_pred = self.image_model(image_emb)     # (batch, num_categories)
        meta_pred = self.metadata_model(metadata)    # (batch, num_categories)
        
        # Weighted average of predictions
        weights = torch.softmax(self.fusion_weights, dim=0)
        combined = (weights[0] * text_pred + 
                   weights[1] * image_pred + 
                   weights[2] * meta_pred)
        
        return combined


# ---- Compare both models ----
torch.manual_seed(42)
batch_size = 8
harm_categories = ['Violence', 'Nudity', 'Self-harm', 'Hate Speech', 'Harassment']

early_model = EarlyFusionHarmDetector()
late_model = LateFusionHarmDetector()

# Simulate features
text = torch.randn(batch_size, 768)
image = torch.randn(batch_size, 512)
meta = torch.randn(batch_size, 49)

early_preds = early_model(text, image, meta)
late_preds = late_model(text, image, meta)

early_params = sum(p.numel() for p in early_model.parameters())
late_params = sum(p.numel() for p in late_model.parameters())

print("=== Model Comparison ===")
print(f"{'Metric':<25} {'Early Fusion':>15} {'Late Fusion':>15}")
print("-" * 55)
print(f"{'Parameters':.<25} {early_params:>15,} {late_params:>15,}")
print(f"{'Output shape':.<25} {str(early_preds.shape):>15} {str(late_preds.shape):>15}")
print(f"{'Cross-modal detection':.<25} {'YES':>15} {'NO':>15}")
print(f"{'Models to maintain':.<25} {'1':>15} {'3':>15}")
print(f"{'Debug ease':.<25} {'Harder':>15} {'Easier':>15}")
print()
print("Early fusion output (first post):")
for cat, prob in zip(harm_categories, early_preds[0].tolist()):
    print(f"  P({cat}): {prob:.4f}")

---

## 8. The Meme Problem: Cross-Modal Detection

This is the killer example that demonstrates why early fusion is essential. Let's simulate how a harmful meme (harmless image + harmless text = harmful combination) fools late fusion but is caught by early fusion.

Think of it like a jigsaw puzzle. Each piece looks innocent by itself -- a blue sky piece, a green grass piece. But when you put them together, they form a picture of something problematic.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# ======================================================================
# Meme Detection: The Case for Early Fusion
#
# Scenario: A meme with a harmless image (cartoon frog) and harmless
# text ("When I see them"), but together they express hate speech.
#
# Late fusion: Each modality analyzed separately -> both seem harmless
# Early fusion: Modalities analyzed together -> cross-modal harm detected
# ======================================================================

np.random.seed(42)

# Simulate 3 types of content
scenarios = [
    {
        'name': 'Normal Post\n(text + food photo)',
        'text_harm': 0.05,
        'image_harm': 0.03,
        'cross_modal_harm': 0.04,  # No cross-modal effect
        'actual_harmful': False,
    },
    {
        'name': 'Violent Video\n(explicit violence)',
        'text_harm': 0.10,
        'image_harm': 0.95,
        'cross_modal_harm': 0.93,  # Image alone is enough
        'actual_harmful': True,
    },
    {
        'name': 'Hateful Meme\n(benign image + benign text\n= harmful together)',
        'text_harm': 0.08,     # Text alone: harmless
        'image_harm': 0.05,    # Image alone: harmless
        'cross_modal_harm': 0.89,  # COMBINED: harmful!
        'actual_harmful': True,
    },
]

fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(len(scenarios))
width = 0.25

# Late fusion: max(text_harm, image_harm) as proxy
late_fusion_scores = [max(s['text_harm'], s['image_harm']) for s in scenarios]
early_fusion_scores = [s['cross_modal_harm'] for s in scenarios]

bars1 = ax.bar(x - width, [s['text_harm'] for s in scenarios], width,
               label='Text Model Only', color='#2196F3', edgecolor='white', linewidth=1.5)
bars2 = ax.bar(x, [s['image_harm'] for s in scenarios], width,
               label='Image Model Only', color='#F44336', edgecolor='white', linewidth=1.5)
bars3 = ax.bar(x + width, early_fusion_scores, width,
               label='Early Fusion (cross-modal)', color='#4CAF50', edgecolor='white', linewidth=1.5)

# Add threshold line
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.7, linewidth=2, label='Detection Threshold (0.5)')

# Labels
ax.set_ylabel('Harm Probability', fontsize=13)
ax.set_title('Why Early Fusion Catches Harmful Memes That Late Fusion Misses',
             fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([s['name'] for s in scenarios], fontsize=11)
ax.legend(fontsize=11, loc='upper left')
ax.set_ylim(0, 1.1)
ax.grid(True, axis='y', alpha=0.3)

# Annotate the meme case
ax.annotate('Late fusion MISSES this!\n(both modalities look safe)',
            xy=(2, 0.08), xytext=(2.5, 0.35),
            fontsize=11, fontweight='bold', color='red',
            arrowprops=dict(arrowstyle='->', color='red', lw=2))

ax.annotate('Early fusion CATCHES it!\n(sees cross-modal interaction)',
            xy=(2 + width, 0.89), xytext=(1.0, 0.75),
            fontsize=11, fontweight='bold', color='green',
            arrowprops=dict(arrowstyle='->', color='green', lw=2))

plt.tight_layout()
plt.show()

print("THE MEME PROBLEM:")
print("  Text alone:  'When I see them at the store'  -> P(hate) = 0.08  (harmless!)")
print("  Image alone: Cartoon frog                    -> P(hate) = 0.05  (harmless!)")
print("  Combined:    Frog + that specific text        -> P(hate) = 0.89  (HARMFUL!)")
print()
print("Late fusion: max(0.08, 0.05) = 0.08  --> MISSED!")
print("Early fusion: cross-modal analysis    = 0.89  --> CAUGHT!")
print()
print("This is THE key argument for early fusion in your interview.")

---

## 9. Feature Importance Analysis

Not all features are created equal. Let's analyze which ones contribute most to detecting each type of harm. This helps us understand the model and prioritize engineering effort.

Think of it like a detective asking: "Which pieces of evidence were most useful in solving the case?"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ======================================================================
# Feature Importance Analysis
#
# In practice, you'd use SHAP values or permutation importance.
# Here we show estimated importance based on domain knowledge and
# published research on harmful content detection systems.
# ======================================================================

feature_groups = ['Text\nContent', 'Image/\nVideo', 'User\nReactions', 'Author\nHistory', 'Context']
harm_categories = ['Violence', 'Nudity', 'Hate Speech', 'Self-harm', 'Harassment']

# Estimated relative importance (rows = harm categories, cols = feature groups)
importance_matrix = np.array([
    [0.15, 0.50, 0.15, 0.10, 0.10],  # Violence: heavily image-dependent
    [0.05, 0.60, 0.15, 0.10, 0.10],  # Nudity: almost entirely image-based
    [0.45, 0.10, 0.20, 0.15, 0.10],  # Hate speech: heavily text-dependent
    [0.40, 0.10, 0.25, 0.15, 0.10],  # Self-harm: text + reactions
    [0.35, 0.05, 0.25, 0.25, 0.10],  # Harassment: text + author history
])

fig, ax = plt.subplots(figsize=(12, 7))
im = ax.imshow(importance_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.6)

# Labels
ax.set_xticks(range(len(feature_groups)))
ax.set_xticklabels(feature_groups, fontsize=11, fontweight='bold')
ax.set_yticks(range(len(harm_categories)))
ax.set_yticklabels(harm_categories, fontsize=11, fontweight='bold')

# Add values
for i in range(len(harm_categories)):
    for j in range(len(feature_groups)):
        val = importance_matrix[i, j]
        color = 'white' if val > 0.35 else 'black'
        ax.text(j, i, f'{val:.0%}', ha='center', va='center', fontsize=12,
                fontweight='bold', color=color)

ax.set_title('Feature Importance by Harm Category\n(Which features matter most for each type of harm?)',
             fontsize=14, fontweight='bold', pad=15)
plt.colorbar(im, ax=ax, label='Relative Importance', shrink=0.8)
plt.tight_layout()
plt.show()

print("Key Insights:")
print("  1. Violence & Nudity: Image features dominate (50-60%)")
print("     -> Invest in strong visual encoders (CLIP, SimCLR)")
print("  2. Hate Speech: Text features dominate (45%)")
print("     -> Invest in multilingual text models (DistilmBERT)")
print("  3. Self-harm: User reactions are very important (25%)")
print("     -> Concerned comments from friends are a strong signal")
print("  4. Harassment: Author history matters a lot (25%)")
print("     -> Repeat offenders are more likely to harass")
print("  5. Context features contribute evenly (~10%) across all types")
print()
print("This is exactly why we need MULTI-TASK learning:")
print("Different harm types rely on different features!")
print("Shared layers learn common patterns, task heads specialize.")

---

## 10. Summary and Interview Key Points

### Feature Engineering Summary (Figure 5.15 from the PDF)

```
Feature Category     | Encoding Method           | Dimension | Key Models
---------------------+---------------------------+-----------+-----------
Text content         | DistilmBERT embedding     | 768       | DistilmBERT
Image/Video          | Pre-trained CNN embedding | 512       | CLIP, SimCLR
Comments (reactions) | DistilmBERT + average     | 768       | Same as text
Reaction counts      | Log-scaled numerical      | 7         | Feature eng.
Author (numerical)   | Log-scaled                | 7         | Feature eng.
Author (categorical) | One-hot + embeddings      | 27        | Embedding layer
Context              | One-hot encoded           | 8         | Feature eng.
---------------------+---------------------------+-----------+-----------
TOTAL FUSED VECTOR                               | 2,097     | Concatenated
```

### Interview Cheat Sheet for This Topic

| Question | Key Phrase |
|----------|------------|
| Why DistilBERT? | "60% faster inference, 97% BERT quality, multilingual support" |
| Why CLIP for images? | "Trained on 400M image-text pairs, understands cross-modal relationships" |
| How do you handle videos? | "Sample key frames uniformly, embed each with CLIP, average embeddings" |
| Why embed comments? | "Comment sentiment is a strong signal; concerned friends flag self-harm" |
| Why one-hot for gender but embedding for city? | "Low cardinality (3 values) = one-hot; high cardinality (10K cities) = learned embedding" |
| Why early fusion? | "Cross-modal harm detection -- memes that are benign per-modality but harmful combined" |
| What is the feature vector size? | "~2,097 dimensions, dominated by text and image embeddings" |

In [ ]:
# Final recap
print("=" * 70)
print("NOTEBOOK 2 SUMMARY: MULTIMODAL FUSION & FEATURE ENGINEERING")
print("=" * 70)
print()
print("FIVE FEATURE CATEGORIES:")
print("  1. Text Content     -> DistilmBERT -> 768-dim embedding")
print("  2. Image/Video      -> CLIP encoder -> 512-dim embedding")
print("  3. User Reactions   -> Comment embeddings + reaction counts")
print("  4. Author Info      -> Numerical (log-scaled) + categorical (embedded)")
print("  5. Context          -> Time bucket + device (one-hot)")
print()
print("FUSION STRATEGY:")
print("  Early Fusion: Concatenate ALL features -> Single 2,097-dim vector")
print("  WHY: Captures cross-modal harm (memes, sarcastic image+text combos)")
print()
print("KEY INSIGHT:")
print("  Different harm types depend on different features.")
print("  Violence needs strong image features.")
print("  Hate speech needs strong text features.")
print("  Self-harm needs strong reaction features.")
print("  -> This is why multi-task learning with shared + specialized layers works best.")
print()
print("NEXT: Notebook 3 covers the multi-task training loop, loss functions,")
print("      evaluation metrics, and serving architecture.")

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)